In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os
print(os.listdir('/content/gdrive/MyDrive/'))

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
['NEW CV (20).pdf', 'NEW CV (19).pdf', 'NEW CV (18).pdf', 'NEW CV (17).pdf', '20201221_170145.jpg', 'IMG-20191231-WA0001.jpg', 'IMG-20190604-WA0001.jpg', 'akash cont.vcf', 'IMG-20220526-WA0003.jpg', 'certificate.pdf', 'IMG-20230227-WA0002.jpg', 'IMG_20230309_233210-01.jpeg', '132CANON.zip', 'old', '7hills', 'Google Finance Investment Tracker (1).gsheet', 'locations for hybrid.gdoc', 'Untitled spreadsheet (4).gsheet', 'Untitled document (6).gdoc', 'GATE.gsheet', 'Screenshot_2024-03-13-23-13-41-21_1c337646f29875672b5a61192b9010f9.jpg', 'Untitled spreadsheet (3).gsheet', 'Akash panda.pdf', 'Akash CV (9).pdf', '76616c59-28e2-4e60-b748-3a8cf4e062a8_payment_confirmation_202406164-073410.png', 'Data Analyst - ats compatible.docx', 'Akash CV (8).pdf', 'Akash CV (7).pdf', 'Akash CV (6).pdf', 'Akash CV (5).pdf', 'Akash CV (4).pdf', 'Akash CV (3).pdf', 'Akash CV (2).p

In [ ]:
import sys
print(sys.path)

['/content', '/env/python', '/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/usr/local/lib/python3.11/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.11/dist-packages/IPython/extensions', '/root/.ipython', '/usr/local/lib/python3.11/dist-packages/setuptools/_vendor']


In [ ]:
import os
print(os.listdir('/content/gdrive/MyDrive/Code'))

['eda.ipynb', 'module_3_CNN_baseline_model.ipynb', '.ipynb_checkpoints', '__pycache__', 'Restaurent_data', 'module_4_CNN_google_bert_augmented.ipynb', 'module_2_preprocessing.ipynb', 'module_2_preprocessing.py', 'preprocessed_data', 'trained_model', 'GANaugmentation.ipynb']


In [ ]:
restaurant_data = pd.read_csv("/content/gdrive/MyDrive/Code/Restaurent_data/augmented_data_restaurant_bert.csv")

In [ ]:
# # Load dataset
# restaurant_data = pd.read_csv("/content/augmented_data_restaurant_bert.csv")

In [ ]:
# Extract text and labels
X = restaurant_data['text']
y = restaurant_data['aspect_category']


In [ ]:
# Tokenization
vocab_size = 10000  # Limit vocabulary size
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X)

In [ ]:
# Convert text to tokenized matrix
X_tokenized = tokenizer.texts_to_matrix(X)  # Convert to numerical representation


In [ ]:
# Pad sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X_tokenized, maxlen=500, padding='post')

In [ ]:
# Encode aspect categories
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)  # One-hot encoding


In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_tokenized, y_categorical, test_size=0.2, random_state=1)

In [ ]:
# def build_generator(latent_dim, output_dim):
#     model = Sequential([
#         Dense(256, input_dim=latent_dim),
#         LeakyReLU(alpha=0.2),
#         BatchNormalization(),
#         Dropout(0.2),
#         Dense(512),
#         LeakyReLU(alpha=0.2),
#         BatchNormalization(),
#         Dropout(0.2),
#         Dense(output_dim, activation='tanh')
#     ])
#     return model
from tensorflow.keras.layers import Dense, LeakyReLU, Dropout, BatchNormalization

def build_generator(latent_dim, output_dim):
    model = Sequential([
        Dense(512, input_dim=latent_dim),
        LeakyReLU(alpha=0.2),
        BatchNormalization(),
        Dropout(0.3),

        Dense(1024),
        LeakyReLU(alpha=0.2),
        BatchNormalization(),
        Dropout(0.3),

        Dense(output_dim, activation='tanh')
    ])
    return model


In [ ]:
def build_discriminator(input_dim):
    model = Sequential([
        Dense(512, input_dim=input_dim),
        LeakyReLU(alpha=0.2),
         Dropout(0.3),
        Dense(256),
        LeakyReLU(alpha=0.2),
         Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])
    return model


In [ ]:
latent_dim = 100  # Noise vector size
output_dim = X_train.shape[1]  # Same as input dimension for the classifier

generator = build_generator(latent_dim, output_dim)
discriminator = build_discriminator(output_dim)

# Make Discriminator non-trainable when training GAN
discriminator.trainable = False

gan = Sequential([generator, discriminator])
gan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [ ]:
!pip install --upgrade numpy
!pip install gensim

  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  A

In [ ]:
!pip install --upgrade --force-reinstall gensim

  Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached smart_open-7.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-1.17.2-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (26.7 MB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (38.6 MB)
Using cached smart_open-7.1.0-py3-none-any.whl (61 kB)
Using cached wrapt-1.17.2-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (83 kB)
  Att

In [ ]:
!pip show numpy
!pip show gensim

Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2023, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyright notice, this list of conditions and the following
       disclaimer in the documentation and/or other materials provided
       with the distribution.

    * Neither the name of the NumPy Developers nor the names of any
       contributors may be used to endorse or promote products derived
       from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYR

In [ ]:
!pip install gensim #installing the gensim library in the notebook instance.
import numpy as np
import gensim.downloader as api

def get_embedding_matrix(tokenizer, embedding_dim=50):
    # Load Pretrained GloVe (50D vectors)
    glove_model = api.load("glove-wiki-gigaword-50")

    vocab_size = min(len(tokenizer.word_index) + 1, 6000)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        embedding_vector = glove_model.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

    return embedding_matrix

In [ ]:
!pip install gensim
import gensim.downloader as api

# Load Pretrained GloVe (50D vectors)
glove_model = api.load("glove-wiki-gigaword-50")

def get_embedding_matrix(tokenizer, embedding_dim=50):
    vocab_size = min(len(tokenizer.word_index) + 1, 6000)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        # Use bracket notation to get the embedding vector
        try:
            embedding_vector = glove_model[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            # Handle cases where the word is not in the GloVe vocabulary
            pass

    return embedding_matrix


# from transformers import BertTokenizer, TFBertModel
# import tensorflow as tf

# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# bert_model = TFBertModel.from_pretrained('bert-base-uncased')

# def bert_embedding(texts):
#     # Ensure texts is a list of strings
#     if isinstance(texts, np.ndarray):
#         texts = texts.tolist()
#     elif isinstance(texts, pd.Series):
#         texts = texts.astype(str).tolist()
#     else:
#         texts = [str(text) for text in texts]

#     tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="tf")
#     embeddings = bert_model(**tokens).last_hidden_state[:, 0, :]
#     return embeddings.numpy()

# # Convert text to BERT embeddings
# # Convert text to BERT embeddings
# # X_train_bert = bert_embedding(X[restaurant_data['text']])  # Use original text data 'X' with corresponding indices from X_train
# # X_test_bert = bert_embedding(X[restaurant_data['text']])   # Use original text data 'X' with corresponding indices from X_test

# X_train_bert = bert_embedding(X.tolist(restaurant_data['text']))  # Pass the list of text values directly to bert_embedding
# X_test_bert = bert_embedding(X.tolist(restaurant_data['text']))  # Pass the list of text values directly to bert_embedding

In [ ]:
# import gensim.downloader as api

# # Load Pretrained GloVe (50D vectors)
# glove_model = api.load("glove-wiki-gigaword-50")

# def get_embedding_matrix(tokenizer, embedding_dim=50):
#     vocab_size = min(len(tokenizer.word_index) + 1, 6000)
#     embedding_matrix = np.zeros((vocab_size, embedding_dim))

#     for word, i in tokenizer.word_index.items():
#         if i >= vocab_size:
#             continue
#         # Use bracket notation to get the embedding vector
#         try:
#             embedding_vector = glove_model[word]  # Changed from glove_model.get(word)
#             embedding_matrix[i] = embedding_vector
#         except KeyError:
#             # Handle cases where the word is not in the GloVe vocabulary
#             pass

#     return embedding_matrix

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Bidirectional, Dropout, Dense # Add imports for LSTM, Embedding, Bidirectional, and Dropout, Dense
from tensorflow.keras.preprocessing.text import Tokenizer # Import Tokenizer
import pandas as pd # Import pandas
import gensim.downloader as api # Import the necessary module to load GloVe
import numpy as np # Import numpy and give it alias np

# Load Pretrained GloVe (50D vectors)
glove_model = api.load("glove-wiki-gigaword-50") # Call api.load to download and load the model


# Define vocab_size before using it in Tokenizer
vocab_size = 10000  # Or any desired vocabulary size

# ... [Code for data loading and extraction, including this line:]
restaurant_data = pd.read_csv("/content/gdrive/MyDrive/Code/Restaurent_data/augmented_data_restaurant_bert.csv")
X = restaurant_data['text']  # Assuming 'text' is your column name
y = restaurant_data['aspect_category']  # Assuming 'aspect_category' is your column name
# ...

# Create a Keras tokenizer and fit it to your data
tokenizer_keras = Tokenizer(num_words=vocab_size)  # vocab_size is already defined
tokenizer_keras.fit_on_texts(X)

def get_embedding_matrix(tokenizer, embedding_dim=50): # Define the function get_embedding_matrix here
    vocab_size = min(len(tokenizer.word_index) + 1, 6000)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        # Use bracket notation to get the embedding vector
        try:
            embedding_vector = glove_model[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            # Handle cases where the word is not in the GloVe vocabulary
            pass

    return embedding_matrix

embedding_dim = 50  # Same as GloVe
embedding_matrix = get_embedding_matrix(tokenizer_keras, embedding_dim) # Pass tokenizer_keras here

# Get the actual vocabulary size
vocab_size = embedding_matrix.shape[0]

classifier = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False),

    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3)),

    Dense(256, activation='relu'), #Dense is imported now
    Dropout(0.3),

    Dense(128, activation='relu'), #Dense is imported now
    Dropout(0.3),

    Dense(5, activation='softmax') #Dense is imported now
])

In [ ]:
# from tensorflow.keras.layers import LSTM, Embedding, Bidirectional

# embedding_dim = 50  # Same as GloVe
# embedding_matrix = get_embedding_matrix(tokenizer, embedding_dim)

# # Get the actual vocabulary size
# vocab_size = embedding_matrix.shape[0]

# classifier = Sequential([
#     Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False), # Use vocab_size here

#     Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
#     Bidirectional(LSTM(64, dropout=0.3)),

#     Dense(256, activation='relu'),
#     Dropout(0.3),

#     Dense(128, activation='relu'),
#     Dropout(0.3),

#     Dense(5, activation='softmax')
# ])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import plot_model

# Load dataset (replace with actual dataset)
df = pd.read_csv("/content/gdrive/MyDrive/Code/Restaurent_data/augmented_data_restaurant_bert.csv")  # Ensure correct file path

# Extract text and labels
X = df["text"].astype(str).tolist()  # Ensure text is in string format
y = df["polarity"].values  # Replace 'label' with the actual column name for sentiment labels

# Tokenize text
vocab_size = 10000  # Define vocabulary size
tokenizer_keras = Tokenizer(num_words=vocab_size)
tokenizer_keras.fit_on_texts(X)

# Convert text to sequences
X_sequences = tokenizer_keras.texts_to_sequences(X)
X_padded = pad_sequences(X_sequences, maxlen=100)  # Adjust maxlen as needed

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)

# Get input shape
input_shape = X_train.shape[1]
print(f"Input shape: {input_shape}")

# Define embedding dimensions
embedding_dim = 50

# Define CNN-BERT Classifier Model
classifier = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=input_shape),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3)),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')
])

# Build classifier model
classifier.build(input_shape=(None, input_shape))

# Plot classifier model structure
plot_model(classifier, to_file="classifier_model.png", show_shapes=True, show_layer_names=True)

# Plot GAN Model (Generator & Discriminator)
try:
    plot_model(generator, to_file="generator_model.png", show_shapes=True, show_layer_names=True)
    plot_model(discriminator, to_file="discriminator_model.png", show_shapes=True, show_layer_names=True)
except NameError:
    print("Error: Generator or Discriminator is not defined. Ensure your GAN model is properly initialized.")


Input shape: 100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# from tensorflow.keras.utils import plot_model

# # Plot the classifier model structure
# plot_model(classifier, to_file="classifier_model.png", show_shapes=True, show_layer_names=True)

# # Plot the GAN structure (Generator & Discriminator)
# plot_model(generator, to_file="generator_model.png", show_shapes=True, show_layer_names=True)
# plot_model(discriminator, to_file="discriminator_model.png", show_shapes=True, show_layer_names=True)


In [ ]:
# classifier.summary()


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU
from tensorflow.keras.optimizers import Adam

# Define the latent space dimension
latent_dim = 100

# Create Generator Model
def build_generator():
    model = Sequential()
    model.add(Dense(128, activation=LeakyReLU(alpha=0.2), input_dim=latent_dim))
    model.add(Dense(256, activation=LeakyReLU(alpha=0.2)))
    model.add(Dense(512, activation=LeakyReLU(alpha=0.2)))
    model.add(Dense(784, activation='tanh'))  # Assuming MNIST dataset (28x28=784)
    return model

# Create Discriminator Model
def build_discriminator():
    model = Sequential()
    model.add(Dense(512, activation=LeakyReLU(alpha=0.2), input_dim=784))
    model.add(Dense(256, activation=LeakyReLU(alpha=0.2)))
    model.add(Dense(128, activation=LeakyReLU(alpha=0.2)))
    model.add(Dense(1, activation='sigmoid'))  # Output probability of real vs fake
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])
    return model

# Create GAN Model (Generator + Discriminator)
def build_gan(generator, discriminator):
    discriminator.trainable = False  # Freeze discriminator while training GAN
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))
    return model

# Load dataset (assuming MNIST)
(X_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
X_train = X_train.reshape(X_train.shape[0], 784).astype('float32') / 127.5 - 1  # Normalize to [-1,1]

# Initialize models
generator = build_generator()
discriminator = build_discriminator()
gan = build_gan(generator, discriminator)

# Train GAN function
def train_gan(epochs, batch_size=128):
    for epoch in range(epochs):
        # Select real samples
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        real_samples = X_train[idx]

        # Generate fake samples
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_samples = generator.predict(noise)

        # Create labels
        real_labels = np.ones((batch_size, 1))
        fake_labels = np.zeros((batch_size, 1))

        # Train Discriminator
        d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # Train Generator (through GAN)
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))

        # Print progress
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, D Loss: {d_loss[0]}, G Loss: {g_loss}")

# Train the GAN
train_gan(epochs=5000, batch_size=256)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step  


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:82: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Streaming output truncated to the last 5000 lines.
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 

In [ ]:
def generate_synthetic_data(num_samples=500):
    noise = np.random.normal(0, 1, (num_samples, latent_dim))
    synthetic_data = generator.predict(noise)
    return synthetic_data

synthetic_samples = generate_synthetic_data(500)


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [ ]:
X_combined = np.vstack((X_train, synthetic_samples))
y_combined = np.vstack((y_train, y_train[:synthetic_samples.shape[0]]))  # Duplicate labels

X_test_final = X_test
y_test_final = y_test
# X_combined = np.vstack((X_train_bert, generate_synthetic_data(500)))
# y_combined = np.vstack((y_train, y_train[:500]))

# classifier.fit(X_combined, y_combined, validation_data=(X_test_bert, y_test), epochs=30, batch_size=128, callbacks=[lr_scheduler])

# X_combined = np.vstack((X_train_bert, generate_synthetic_data(500)))
# y_combined = np.vstack((y_train, y_train[:500]))

# classifier.fit(X_combined, y_combined, validation_data=(X_test_bert, y_test), epochs=30, batch_size=128, callbacks=[lr_scheduler])


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 4870 and the array at index 1 has size 500

In [ ]:
from tensorflow.keras.layers import LSTM, Bidirectional, Attention, Embedding, Input

def build_advanced_classifier(input_dim, output_dim):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),

        Dense(256, activation='relu'),
        Dropout(0.3),

        Dense(output_dim, activation='softmax')
    ])
    return model

# Assuming you want to use X_train instead of X_train_bert
classifier = build_advanced_classifier(X_train.shape[1], 5)
# If you intended to use BERT, uncomment and execute the BERT embedding code
# and then use X_train_bert.shape[1] here

In [ ]:
# GELU
classifier = Sequential([
    Dense(512, activation='relu', input_shape=(output_dim,)),
    Dropout(0.3), # Dropout is now recognized
    Dense(256, activation='relu'),
    Dropout(0.3), # Dropout is now recognized
    Dense(5, activation='softmax')
])

classifier.compile(loss='categorical_crossentropy', optimizer=Adam(0.0005), metrics=['accuracy'])

In [ ]:
# from tensorflow.keras.layers import LSTM, Bidirectional, Attention, Embedding, Input

# def build_advanced_classifier(input_dim, output_dim):
#     model = Sequential([
#         Dense(512, activation='relu', input_shape=(input_dim,)),
#         Dropout(0.3),

#         Dense(256, activation='relu'),
#         Dropout(0.3),

#         Dense(output_dim, activation='softmax')
#     ])
#     return model

# # Assuming you want to use X_train instead of X_train_bert
# classifier = build_advanced_classifier(X_train.shape[1], 5)
# # If you intended to use BERT, uncomment and execute the BERT embedding code
# # and then use X_train_bert.shape[1] here

In [ ]:
# build_advanced_classifier.summary()


In [ ]:
# history = classifier.fit(X_combined, y_combined, validation_data=(X_test_final, y_test_final), epochs=25, verbose=1)


In [ ]:
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau

optimizer = AdamW(learning_rate=0.0005)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

classifier.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])


In [ ]:
history = classifier.fit(X_combined, y_combined, validation_data=(X_test_final, y_test_final), epochs=25, verbose=1)

In [ ]:
predictions = classifier.predict(X_test_final)
predicted_labels = np.argmax(predictions, axis=1)
predicted_cat = label_encoder.inverse_transform(predicted_labels)

from sklearn.metrics import classification_report
print(classification_report(label_encoder.inverse_transform(np.argmax(y_test_final, axis=1)), predicted_cat))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline

def smooth_curve(x, y, num_points=300):
    """Interpolate to create a smooth curve."""
    x_new = np.linspace(x.min(), x.max(), num_points)
    spline = make_interp_spline(x, y, k=3)  # Cubic spline interpolation
    y_smooth = spline(x_new)
    return x_new, y_smooth

def plot_accuracy(history, miny=None):
    acc = np.array(history.history['accuracy'])
    test_acc = np.array(history.history['val_accuracy'])
    epochs = np.array(range(len(acc)))

    # Get smooth curves
    smooth_epochs, smooth_acc = smooth_curve(epochs, acc)
    _, smooth_test_acc = smooth_curve(epochs, test_acc)

    plt.plot(smooth_epochs, smooth_acc, label='Training Accuracy')
    plt.plot(smooth_epochs, smooth_test_acc, label='Validation Accuracy')

    if miny:
        plt.ylim(miny, 1.0)

    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

# Call the function
plot_accuracy(history)

In [ ]:
# code borrowed from Microsoft Malware Detection Assignment
def plot_confusion_matrix(test_y, predict_y ,labels):
    C = confusion_matrix(test_y, predict_y)      # calculation of confusion matrix
    print("Number of misclassified points ",(len(test_y)-np.trace(C))/len(test_y))    # number of misclassified points while predicting y

    A =(((C.T)/(C.sum(axis=1))).T)
    B =(C/C.sum(axis=0))

    labels = labels
    cmap=sns.light_palette("purple")
    # representing A in heatmap format
    print("-"*50, "Confusion matrix", "-"*50)
    plt.figure(figsize=(10,5))
    sns.heatmap(C, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.show()

    print("-"*50, "Precision matrix", "-"*50)
    plt.figure(figsize=(10,5))
    sns.heatmap(B, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.show()
    print("Sum of columns in precision matrix",B.sum(axis=0))

    # representing B in heatmap format
    print("-"*50, "Recall matrix" , "-"*50)
    plt.figure(figsize=(10,5))
    sns.heatmap(A, annot=True, cmap=cmap, fmt=".3f", xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted Class')
    plt.ylabel('Original Class')
    plt.show()
    print("Sum of rows in precision matrix",A.sum(axis=1))

In [ ]:
# plot_confusion_matrix(y_test,  predicted_cat, catagories)
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(label_encoder.inverse_transform(np.argmax(y_test_final, axis=1)), predicted_cat)
# Plot confusion matrix
plt.figure(figsize=(8, 6))
# Change 'purple' to a valid colormap name like 'Purples' or 'viridis'
sns.heatmap(cm, annot=True, fmt="d",cmap=sns.light_palette("purple"),  xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extract training history
plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Model Accuracy')

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Model Loss')

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def smooth_curve(points, factor=0.8):
    """Smooths the curve using exponential moving average."""
    smoothed_points = []
    for point in points:
        if smoothed_points:
            smoothed_points.append(smoothed_points[-1] * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

# Extract training history
train_acc = smooth_curve(history.history['accuracy'])
val_acc = smooth_curve(history.history['val_accuracy'])
train_loss = smooth_curve(history.history['loss'])
val_loss = smooth_curve(history.history['val_loss'])

plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Smoothed Train Accuracy')
plt.plot(val_acc, label='Smoothed Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Smoothed Model Accuracy')

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(train_loss, label='Smoothed Train Loss')
plt.plot(val_loss, label='Smoothed Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Smoothed Model Loss')

plt.show()


In [ ]:
# from scipy.signal import savgol_filter

# # Apply Savitzky-Golay filter for smoothing
# window_size = 5  # Adjust for more/less smoothing
# poly_order = 2   # Polynomial order for smoothing

# train_acc_smooth = savgol_filter(history.history['accuracy'], window_size, poly_order)
# val_acc_smooth = savgol_filter(history.history['val_accuracy'], window_size, poly_order)
# train_loss_smooth = savgol_filter(history.history['loss'], window_size, poly_order)
# val_loss_smooth = savgol_filter(history.history['val_loss'], window_size, poly_order)

# plt.figure(figsize=(12, 5))

# # Accuracy Plot
# plt.subplot(1, 2, 1)
# plt.plot(train_acc_smooth, label='Smoothed Train Accuracy')
# plt.plot(val_acc_smooth, label='Smoothed Validation Accuracy')
# plt.xlabel('Epochs')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.title('Smoothed Model Accuracy (Savitzky-Golay)')

# # Loss Plot
# plt.subplot(1, 2, 2)
# plt.plot(train_loss_smooth, label='Smoothed Train Loss')
# plt.plot(val_loss_smooth, label='Smoothed Validation Loss')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()
# plt.title('Smoothed Model Loss (Savitzky-Golay)')

# plt.show()


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# # Load dataset
# restaurant_data = pd.read_csv("/content/augmented_data_restaurant.csv")

# Extract text and labels
X = restaurant_data['text']
y = restaurant_data['aspect_category']

# Tokenization
vocab_size = 6000  # Limit vocabulary size
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X)

# Convert text to tokenized matrix
X_tokenized = tokenizer.texts_to_matrix(X)  # Convert to numerical representation

# Encode aspect categories
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)  # One-hot encoding

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_tokenized, y_categorical, test_size=0.2, random_state=1)


In [ ]:
def build_generator(latent_dim, output_dim):
    model = Sequential([
        Dense(256, input_dim=latent_dim),
        LeakyReLU(alpha=0.2),
        BatchNormalization(),
        Dense(512),
        LeakyReLU(alpha=0.2),
        BatchNormalization(),
        Dense(output_dim, activation='tanh')
    ])
    return model


In [ ]:
def build_discriminator(input_dim):
    model = Sequential([
        Dense(512, input_dim=input_dim),
        LeakyReLU(alpha=0.2),
        Dense(256),
        LeakyReLU(alpha=0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])
    return model


In [ ]:
latent_dim = 100  # Noise vector size
output_dim = X_train.shape[1]  # Same as input dimension for the classifier

generator = build_generator(latent_dim, output_dim)
discriminator = build_discriminator(output_dim)

# Make Discriminator non-trainable when training GAN
discriminator.trainable = False

gan = Sequential([generator, discriminator])
gan.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))


In [ ]:
def train_gan(epochs, batch_size=128):
    for epoch in range(epochs):
        # Select real samples
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        real_samples = X_train[idx]

        # Generate fake samples
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_samples = generator.predict(noise)

        # Create labels
        real_labels = np.ones((batch_size, 1))
        fake_labels = np.zeros((batch_size, 1))

        # Train Discriminator
        d_loss_real = discriminator.train_on_batch(real_samples, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # Train Generator (through GAN)
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))

        # Print progress
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, D Loss: {d_loss[0]}, G Loss: {g_loss}")

train_gan(epochs=5000, batch_size=128)


In [ ]:
def generate_synthetic_data(num_samples=500):
    noise = np.random.normal(0, 1, (num_samples, latent_dim))
    synthetic_data = generator.predict(noise)
    return synthetic_data

synthetic_samples = generate_synthetic_data(500)


In [ ]:
X_combined = np.vstack((X_train, synthetic_samples))
y_combined = np.vstack((y_train, y_train[:synthetic_samples.shape[0]]))  # Duplicate labels

X_test_final = X_test
y_test_final = y_test


In [ ]:
classifier = Sequential([
    Dense(512, activation='relu', input_shape=(output_dim,)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')
])

classifier.compile(loss='categorical_crossentropy', optimizer=Adam(0.0005), metrics=['accuracy'])


In [ ]:
from tensorflow.keras.utils import plot_model

# Plot the classifier model structure
plot_model(classifier, to_file="classifier_model.png", show_shapes=True, show_layer_names=True)

# Plot the GAN structure (Generator & Discriminator)
plot_model(generator, to_file="generator_model.png", show_shapes=True, show_layer_names=True)
plot_model(discriminator, to_file="discriminator_model.png", show_shapes=True, show_layer_names=True)


In [ ]:
history = classifier.fit(X_combined, y_combined, validation_data=(X_test_final, y_test_final), epochs=25, verbose=1)


In [ ]:
predictions = classifier.predict(X_test_final)
predicted_labels = np.argmax(predictions, axis=1)
predicted_cat = label_encoder.inverse_transform(predicted_labels)

from sklearn.metrics import classification_report
print(classification_report(label_encoder.inverse_transform(np.argmax(y_test_final, axis=1)), predicted_cat))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def smooth_curve(points, factor=0.8):
    """Smooths the curve using exponential moving average."""
    smoothed_points = []
    for point in points:
        if smoothed_points:
            smoothed_points.append(smoothed_points[-1] * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

# Extract training history
train_acc = smooth_curve(history.history['accuracy'])
val_acc = smooth_curve(history.history['val_accuracy'])
train_loss = smooth_curve(history.history['loss'])
val_loss = smooth_curve(history.history['val_loss'])

plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Smoothed Train Accuracy')
plt.plot(val_acc, label='Smoothed Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Smoothed Model Accuracy')

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(train_loss, label='Smoothed Train Loss')
plt.plot(val_loss, label='Smoothed Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Smoothed Model Loss')

plt.show()


In [ ]:
# plot_confusion_matrix(y_test,  predicted_cat, catagories)
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(label_encoder.inverse_transform(np.argmax(y_test_final, axis=1)), predicted_cat)
# Plot confusion matrix
plt.figure(figsize=(8, 6))
# Change 'purple' to a valid colormap name like 'Purples' or 'viridis'
sns.heatmap(cm, annot=True, fmt="d",cmap=sns.light_palette("purple"),  xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()